# 01 — Plant Flowering Data (Small-Scale Testing)

Loads the top 50 plant species by observation frequency from PhenoField.
Used for initial go/no-go pipeline testing only — not for final results.

**Output:** `plant_flowering_events.parquet` (top 50 species, ~208k records)

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path("/scratch/ariana.l")
PPE_DIR     = BASE / "ppe-outputs" / "opportunity_surface"
OUT_DIR     = BASE / "Plant Pollinator Initial Analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ───────────────────────────────────────────────────────────────
TOP_N       = 50
BIN_SIZE    = 0.5

print("Paths OK")

In [ ]:
# Load full PhenoField dataset to get species observation counts
print("Scanning PhenoField parquet files for species counts...")

dataset = ds.dataset(str(PPE_DIR), format="parquet")
df_all = dataset.to_table(columns=["species"]).to_pandas()

species_counts = df_all["species"].value_counts()
print(f"  Total species: {len(species_counts):,}")
print(f"  Top 5:\n{species_counts.head()}")

In [ ]:
# Select top 50 species by observation frequency (ranks 2-51, skip rank 1)
# Rank 1 is excluded as it corresponds to an unknown/unresolved species label
top_species = species_counts.iloc[1:TOP_N + 1].index.tolist()

print(f"Selected {len(top_species)} target species")
print(f"  Sample: {top_species[:5]}")

In [ ]:
# Load full records for top 50 species only
print("Loading flowering records for top 50 species...")

dataset = ds.dataset(str(PPE_DIR), format="parquet")
df = dataset.to_table(
    columns=["species", "lat", "lon", "doy", "year"],
    filter=ds.field("species").isin(top_species)
).to_pandas()

print(f"  Records loaded: {len(df):,}")
print(f"  Species: {df['species'].nunique()}")
df.head()

In [ ]:
# Add spatial bins
df["lat_bin"] = (np.floor(df["lat"] / BIN_SIZE) * BIN_SIZE).round(1)
df["lon_bin"] = (np.floor(df["lon"] / BIN_SIZE) * BIN_SIZE).round(1)
df["bin"]     = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

# Add week
df["week"] = ((df["doy"].astype(int) - 1) // 7).clip(0, 51)

print(f"  Unique bins: {df['bin'].nunique():,}")
print(f"  Week range: {df['week'].min()} – {df['week'].max()}")
df[["species", "lat", "lon", "bin", "week"]].head()

In [ ]:
# Save
out_path = OUT_DIR / "plant_flowering_events.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape: {df.shape}")